In [1]:
!nvidia-smi

Sun Jul 26 13:42:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
from zipfile import ZipFile

ARCHIVE_PATH = Path('/content/drive/MyDrive/[ICLR] Embedding KD/ICLR-MDD-new_idea_2.zip')
EXTRACT_ROOT = Path('/content')
EXPECTED_PROJECT_DIR = EXTRACT_ROOT / 'ICLR-MDD-new_idea_2'

assert ARCHIVE_PATH.is_file(), f'Archive not found: {ARCHIVE_PATH}'

def is_tmkd_repo(path):
    return path.is_dir() and (path / 'main.py').is_file() and (path / 'scripts' / 'train_tmkd.sh').is_file()

if not is_tmkd_repo(EXPECTED_PROJECT_DIR):
    print(f'Extracting {ARCHIVE_PATH} to {EXTRACT_ROOT} ...')
    with ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(EXTRACT_ROOT)

# Accept either the expected archive root or another extracted directory
# containing this repository's main.py and TMKD training script.
project_candidates = [EXPECTED_PROJECT_DIR]
project_candidates.extend(
    path.parent
    for path in EXTRACT_ROOT.glob('*/main.py')
    if (path.parent / 'scripts' / 'train_tmkd.sh').is_file()
)
PROJECT_DIR = next(
    (path.resolve() for path in project_candidates if is_tmkd_repo(path)),
    None,
)
assert PROJECT_DIR is not None, (
    'Could not find the extracted repo under /content. '
    'Check the ZIP root directory and confirm scripts/train_tmkd.sh is included.'
)
print(f'Project directory: {PROJECT_DIR}')

Project directory: /content/ICLR-MDD-new_idea_2


In [4]:
%cd $PROJECT_DIR

/content/ICLR-MDD-new_idea_2


In [5]:
# Remove stale cached teacher embeddings before training
!rm -rf cache
!mkdir -p cache
!echo "Removed cache/"

Removed cache/


In [6]:
!pip install -r requirements.txt

In [7]:
# TMKD primary run: full batch kernel with the fixed global coefficient lambda=1.
# The script expects to run from scripts/, so all paths below are relative to that directory.
!test -f data/merged_9_data_3k_each_ver2.csv || (echo 'Missing data/merged_9_data_3k_each_ver2.csv. Re-run the unzip cell or check the archive.' && false)
!mkdir -p checkpoints/tmkd_full_lambda1 analysis/tmkd_full_lambda1
!bash -c 'set -o pipefail; cd scripts && TRAIN_DATA="../data/merged_9_data_3k_each_ver2.csv" STUDENT_MODEL="google-bert/bert-base-uncased" TEACHER_MODEL="Qwen/Qwen3-Embedding-4B" BATCH_SIZE=16 EPOCHS=5 LR=1e-5 MAX_LENGTH=256 SAVE_DIR="../checkpoints/tmkd_full_lambda1" LAMBDA_TMKD=1.0 TMKD_BLOCK_SIZE=512 TMKD_MODE=full bash train_tmkd.sh 2>&1 | tee "../analysis/tmkd_full_lambda1/train.log"'

Training with TMKD method

Configuration for TMKD method:
  task_type                 : pair_cls
  max_length                : 256
  batch_size                : 16
  epochs                    : 5
  learning_rate             : 1e-05
  min_lr                    : 1e-06
  warmup_ratio              : 0.1
  w_task                    : 1.0
  alpha_dtw                 : 0.5
  w_cls                     : 1.0
  temperature               : 0.05
  student_model_name        : google-bert/bert-base-uncased
  teacher_model_name        : Qwen/Qwen3-Embedding-4B
  teacher_dtype             : bfloat16
  student_special_token     : ##
  teacher_special_token     : _
  train_data_path           : ../data/merged_9_data_3k_each_ver2.csv
  eval_data_path            : None
  num_workers               : 2
  distill_method            : tmkd
  save_dir                  : ../checkpoints/tmkd_full_lambda1
  save_every                : 1
  save_best                 : True
  debug_align               : False
  eval

In [8]:
# Run validation first, then reuse validation-selected pair thresholds on the test sets.
import re
from pathlib import Path

import torch
from transformers import AutoModel, AutoTokenizer

from src.evaluation.evaluation_automodel import (
    eval_classification_task,
    eval_pair_task,
    eval_sts_task,
    eval_cls_tasks,
    eval_pair_tasks,
    eval_sts_tasks,
    test_cls_tasks,
    test_pair_tasks,
    test_sts_tasks,
)

checkpoint_dir = Path("checkpoints/tmkd_full_lambda1")
best_checkpoint = checkpoint_dir / "best_model.pt"

if best_checkpoint.exists():
    checkpoint_path = best_checkpoint
else:
    checkpoints = sorted(
        checkpoint_dir.glob("checkpoint_epoch_*.pt"),
        key=lambda p: int(re.search(r"checkpoint_epoch_(\d+)", p.stem).group(1)),
    )
    assert checkpoints, f"No checkpoint found in {checkpoint_dir}"
    checkpoint_path = checkpoints[-1]

print(f"Loading checkpoint: {checkpoint_path}")

try:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

cfg = checkpoint.get("config", {})
student_model_name = cfg.get("student_model_name", "google-bert/bert-base-uncased")
print(f"Student model: {student_model_name}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModel.from_pretrained(student_model_name)
tokenizer = AutoTokenizer.from_pretrained(student_model_name, use_fast=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("\n" + "=" * 80 + "\nValidation benchmarks\n" + "=" * 80)
validation_classification = eval_classification_task(model, eval_cls_tasks, tokenizer)
validation_pair, pair_thresholds = eval_pair_task(model, eval_pair_tasks, tokenizer)
validation_sts = eval_sts_task(model, eval_sts_tasks, tokenizer)

print("\n" + "=" * 80 + "\nTest benchmarks\n" + "=" * 80)
test_classification = eval_classification_task(model, test_cls_tasks, tokenizer)
test_pair, _ = eval_pair_task(model, test_pair_tasks, tokenizer, thresholds=pair_thresholds)
test_sts = eval_sts_task(model, test_sts_tasks, tokenizer)

print("\nAll benchmark evaluations finished.")

AssertionError: No checkpoint found in checkpoints/tmkd_full_lambda1